In [1]:
import copy
import numpy as np
import sympy as sp

import mrmustard.lab as mr
import mrmustard.physics as mrph
import mrmustard.math as mrmath
from mrmustard import settings as mrsettings
from thewalrus.decompositions import blochmessiah

from generalized_photon_catalysis.quadratic_preparation import generalized_e2_preparation, verify_generalized_e2_preparation
from generalized_photon_catalysis.preparation_circuit import get_product_form_preparation_circuit, mra, mra_dag
from photon_catalysis.utils import state_array_to_dict, state_to_string

mrsettings.AUTOSHAPE_MAX = 50
mrsettings.DEFAULT_FOCK_SIZE = 50

In [2]:
def get_prob_fid(target, state):
    prob = state.L2_norm
    state = state.normalize()
    fid = target.fidelity(state)
    return abs(prob), abs(fid)

In [44]:
counts = 60
with open('figures/squeezed-sub.csv', 'w') as f:
    f.write('r,theta,fid1,prob1,fid2,prob2\n')
    for r in np.linspace(0.01, 4.0, counts):
        S = mr.SqueezedVacuum(1, r)
        target = S >> mra(1)
        target = target.normalize()

        for theta in np.linspace(0.01, np.pi / 2, counts):
            ancilla = 2
            U = mr.BSgate((1, ancilla), theta)

            result = (mr.Vacuum(ancilla) >> S) >> U >> mr.Number(ancilla, 1).dual
            prob1, fid1 = get_prob_fid(target, result)

            c = np.tanh(r) / (np.cos(theta)**2)                
            if c < 1:
                r_prime = np.arctanh(c)
                S_prime = mr.SqueezedVacuum(1, r_prime)
                result = (mr.Vacuum(ancilla) >> S_prime) >> U >> mr.Number(ancilla, 1).dual
                prob2, fid2 = get_prob_fid(target, result)
                assert fid2 > 0.99, f'Expected perfect fidelity, got {fid2}'
            else:
                fid2 = 'nan'
                prob2 = 'nan'

            f.write(f'{r},{theta},{fid1},{prob1},{fid2},{prob2}\n')



/home/andrew/Documents/Sorbonne/M2_2/catalysis-experimental/.venv/lib/python3.10/site-packages/mrmustard/physics/ansatz/array_ansatz.py:260: UserWarning: The fock array is being padded with zeros. Is this really necessary?
  warn(
/home/andrew/Documents/Sorbonne/M2_2/catalysis-experimental/.venv/lib/python3.10/site-packages/mrmustard/physics/ansatz/array_ansatz.py:397: RuntimeWarning: invalid value encountered in divide
  return ArrayAnsatz(array=self.array / other, batch_dims=self.batch_dims)
